# Notebook 02: Análisis Exploratorio de Datos (EDA) y Perfilado Automatizado
**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Fase PDCO**: DEVELOPMENT $\rightarrow$ CONTROL | **Capa**: Bronze Data Profiling  
**Estándar**: SWEBOK v4 / DAMA-DMBOK 2 (Data Quality Dimensions)  

---
### Propósito del Diagnóstico
Inspeccionar cuantitativamente las extracciones crudas de `data/RAW/`:
1. **Completitud**: Tasas de missingness relativo por columna.
2. **Validez y Rango**: Percentiles 1% y 99% para detectar anomalías o precios espurios.
3. **Continuidad Temporal**: Gaps y lagunas temporales por municipio o estación meteorológica.
4. **Cardinalidad**: Catálogo preliminar de productos, insumos y variedades.

In [8]:
# Celda 1: Setup e Importaciones Desacopladas
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from src.utils.logger import get_logger
from src.eda.profiler import profile_dataframe, temporal_coverage, cardinality_matrix
from src.ingestion.landing_manager import LandingManager

logger = get_logger("notebooks.02_eda")
logger.info("Iniciando notebook de EDA y perfilado de datos...")
landing = LandingManager()

2026-10-07T19:29:34-0500 | INFO     | agrostats.notebooks.02_eda:<module>:16 | Iniciando notebook de EDA y perfilado de datos...


In [9]:
# Celda 2: Inspección del Manifiesto Bronze y Carga de Datasets
manifest = landing.load_manifest()
datasets_info = manifest.get("datasets", {})
print(f"Total de datasets registrados en Bronze: {len(datasets_info)}")

loaded_dfs = {}
for key, meta in datasets_info.items():
    fpath = Path(meta["file_path"])
    if fpath.exists():
        loaded_dfs[key] = pd.read_parquet(fpath)
        print(f" -> Cargado: {key} ({len(loaded_dfs[key])} filas)")

Total de datasets registrados en Bronze: 11
 -> Cargado: ideam/telemetria_sensores.parquet (5000 filas)
 -> Cargado: ideam/pluviometria_estaciones.parquet (5000 filas)
 -> Cargado: ideam/temperatura_media.parquet (5000 filas)
 -> Cargado: ideam/temperatura_maxima.parquet (5000 filas)
 -> Cargado: ideam/temperatura_minima.parquet (5000 filas)
 -> Cargado: ideam/radiacion_solar.parquet (169 filas)
 -> Cargado: ideam/normales_spi.parquet (5000 filas)
 -> Cargado: dane/sipsa_insumos_precios.parquet (93 filas)
 -> Cargado: dane/sipsa_alimentos_balanceados.parquet (68 filas)
 -> Cargado: upra/upra_exportaciones_bioinsumos.parquet (5000 filas)
 -> Cargado: upra/upra_eva_historico.parquet (5000 filas)


In [10]:
# Celda 3: Perfilado Estadístico y Missingness por Dataset
print("=== PERFILADO DE COMPLETITUD Y TIPOS DE DATOS ===")
for name, df in loaded_dfs.items():
    print(f"\n--- Perfil de: {name} ---")
    profile = profile_dataframe(df)
    print(f"Filas: {profile['total_rows']} | Columnas: {profile['total_columns']}")
    for col, col_info in profile["columns"].items():
        null_pct = col_info["null_percentage"]
        dtype = col_info["dtype"]
        alert = " [ALERTA >20% NULOS]" if null_pct > 20 else ""
        print(f"  * {col:<25} ({dtype:<10}): {null_pct:>6.2f}% nulos ({col_info['null_count']} filas){alert}")

2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 13 columnas, 1566.15 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 12 columnas, 1055.77 KB


2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 12 columnas, 1087.49 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 12 columnas, 1089.97 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 12 columnas, 1090.29 KB


=== PERFILADO DE COMPLETITUD Y TIPOS DE DATOS ===

--- Perfil de: ideam/telemetria_sensores.parquet ---
Filas: 5000 | Columnas: 13
  * codigoestacion            (str       ):   0.00% nulos (0 filas)
  * codigosensor              (str       ):   0.00% nulos (0 filas)
  * fechaobservacion          (str       ):   0.00% nulos (0 filas)
  * valorobservado            (str       ):   0.00% nulos (0 filas)
  * nombreestacion            (str       ):   0.00% nulos (0 filas)
  * departamento              (str       ):   0.00% nulos (0 filas)
  * municipio                 (str       ):   0.00% nulos (0 filas)
  * zonahidrografica          (str       ):   0.00% nulos (0 filas)
  * latitud                   (str       ):   0.00% nulos (0 filas)
  * longitud                  (str       ):   0.00% nulos (0 filas)
  * descripcionsensor         (str       ):   0.28% nulos (14 filas)
  * unidadmedida              (str       ):   0.28% nulos (14 filas)
  * entidad                   (str       ):   0.00%

2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 169 filas, 20 columnas, 48.40 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 24 columnas, 1592.83 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 93 filas, 58 columnas, 73.69 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 68 filas, 6 columnas, 6.90 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 13 columnas, 1884.43 KB


Filas: 5000 | Columnas: 12
  * codigoestacion            (str       ):   0.00% nulos (0 filas)
  * codigosensor              (str       ):   0.00% nulos (0 filas)
  * fechaobservacion          (str       ):   0.00% nulos (0 filas)
  * valorobservado            (str       ):   0.00% nulos (0 filas)
  * nombreestacion            (str       ):   0.00% nulos (0 filas)
  * departamento              (str       ):   0.00% nulos (0 filas)
  * municipio                 (str       ):   0.00% nulos (0 filas)
  * zonahidrografica          (str       ):   0.00% nulos (0 filas)
  * latitud                   (str       ):   0.00% nulos (0 filas)
  * longitud                  (str       ):   0.00% nulos (0 filas)
  * descripcionsensor         (str       ):   0.00% nulos (0 filas)
  * unidadmedida              (str       ):   0.00% nulos (0 filas)

--- Perfil de: ideam/radiacion_solar.parquet ---
Filas: 169 | Columnas: 20
  * estacion                  (str       ):   0.00% nulos (0 filas)
  * departame

2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 17 columnas, 1242.86 KB


  * cod_depto                 (str       ):   0.00% nulos (0 filas)
  * producto                  (str       ):   0.00% nulos (0 filas)
  * descripcion_partida4_dig  (str       ):   0.00% nulos (0 filas)
  * descripcion_partida4_dig_1 (str       ):   0.00% nulos (0 filas)
  * partida                   (str       ):   0.00% nulos (0 filas)
  * tradici_n_producto        (str       ):   0.00% nulos (0 filas)
  * exportaciones_en_valor_usd (str       ):   0.00% nulos (0 filas)
  * exportaciones_en_volumen  (str       ):   0.00% nulos (0 filas)

--- Perfil de: upra/upra_eva_historico.parquet ---
Filas: 5000 | Columnas: 17
  * c_d_dep                   (str       ):   0.00% nulos (0 filas)
  * departamento              (str       ):   0.00% nulos (0 filas)
  * c_d_mun                   (str       ):   0.00% nulos (0 filas)
  * municipio                 (str       ):   0.02% nulos (1 filas)
  * grupo_de_cultivo          (str       ):   0.00% nulos (0 filas)
  * subgrupo_de_cultivo       (str 

In [11]:
# Celda 4: Análisis de Rango y Percentiles Extremos (1% y 99%)
print("=== INSPECCIÓN DE RANGOS NUMÉRICOS Y ANOMALÍAS ===")
for name, df in loaded_dfs.items():
    profile = profile_dataframe(df)
    numeric_cols = [c for c, info in profile["columns"].items() if "numeric_stats" in info]
    if numeric_cols:
        print(f"\nVariables numéricas en {name}:")
        for col in numeric_cols:
            stats = profile["columns"][col]["numeric_stats"]
            print(f"  - {col}:")
            print(f"      Media: {stats['mean']} | Mediana: {stats['median']} | Desv. Est: {stats['std']}")
            print(f"      Mín: {stats['min']} | P01: {stats['p01']} | P99: {stats['p99']} | Máx: {stats['max']}")
            print(f"      Outliers IQR detectados: {stats['outliers_iqr_count']}")

2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 13 columnas, 1566.15 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 12 columnas, 1055.77 KB


=== INSPECCIÓN DE RANGOS NUMÉRICOS Y ANOMALÍAS ===


2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 12 columnas, 1087.49 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 12 columnas, 1089.97 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 12 columnas, 1090.29 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 169 filas, 20 columnas, 48.40 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 5000 filas, 24 columnas, 1592.83 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 93 filas, 58 columnas, 73.69 KB
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:profile_dataframe:77 | Perfilado completado: 68 filas, 6 columnas, 6.90 KB
2026-10-07T19:29:34

In [12]:
# Celda 5: Diagnóstico de Continuidad Temporal y Gaps
print("=== AUDITORÍA DE CONTINUIDAD TEMPORAL ===")
for name, df in loaded_dfs.items():
    date_cols = [c for c in df.columns if any(k in c.lower() for k in ["fecha", "date", "anio", "mes"])]
    if date_cols:
        col_target = date_cols[0]
        cov = temporal_coverage(df, date_column=col_target, freq="MS")
        print(f"\nDataset: {name} (columna: {col_target})")
        print(f"  Rango: {cov['earliest_date']} a {cov['latest_date']}")
        print(f"  Periodos esperados: {cov['expected_periods_count']} | Observados: {cov['actual_periods_count']}")
        if cov["has_gaps"]:
            print(f"  ¡GAPS DETECTADOS! {cov['missing_periods_count']} periodos ausentes: {cov['missing_periods'][:5]}...")
        else:
            print("  Continuidad temporal perfecta (sin lagunas).")

=== AUDITORÍA DE CONTINUIDAD TEMPORAL ===


2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:temporal_coverage:137 | Cobertura temporal: de 2026-10-07 06:59:00 a 2026-10-07 12:57:00. Gaps detectados: False
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:temporal_coverage:137 | Cobertura temporal: de 2004-11-19 01:45:00 a 2019-08-27 20:20:00. Gaps detectados: True
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:temporal_coverage:137 | Cobertura temporal: de 2019-07-01 00:00:00 a 2019-07-01 23:59:00. Gaps detectados: False
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:temporal_coverage:137 | Cobertura temporal: de 2001-02-19 15:08:03 a 2019-08-27 20:00:00. Gaps detectados: True



Dataset: ideam/telemetria_sensores.parquet (columna: fechaobservacion)
  Rango: 2026-10-07 06:59:00 a 2026-10-07 12:57:00
  Periodos esperados: 0 | Observados: 1
  Continuidad temporal perfecta (sin lagunas).

Dataset: ideam/pluviometria_estaciones.parquet (columna: fechaobservacion)
  Rango: 2004-11-19 01:45:00 a 2019-08-27 20:20:00
  Periodos esperados: 177 | Observados: 176
  ¡GAPS DETECTADOS! 2 periodos ausentes: ['2004-12', '2005-01']...

Dataset: ideam/temperatura_media.parquet (columna: fechaobservacion)
  Rango: 2019-07-01 00:00:00 a 2019-07-01 23:59:00
  Periodos esperados: 1 | Observados: 1
  Continuidad temporal perfecta (sin lagunas).


2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:temporal_coverage:137 | Cobertura temporal: de 2001-02-17 17:39:03 a 2019-08-27 14:00:00. Gaps detectados: True
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:temporal_coverage:137 | Cobertura temporal: de 2018-12-01 00:00:00 a 2026-08-01 00:00:00. Gaps detectados: False
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:temporal_coverage:137 | Cobertura temporal: de 2021-01-01 00:00:00 a 2026-08-01 00:00:00. Gaps detectados: False
2026-10-07T19:29:34-0500 | INFO     | agrostats.eda.profiler:temporal_coverage:137 | Cobertura temporal: de 2021-12-03 00:00:00 a 2021-12-03 00:00:00. Gaps detectados: False



Dataset: ideam/temperatura_maxima.parquet (columna: fechaobservacion)
  Rango: 2001-02-19 15:08:03 a 2019-08-27 20:00:00
  Periodos esperados: 222 | Observados: 179
  ¡GAPS DETECTADOS! 44 periodos ausentes: ['2001-03', '2001-04', '2001-05', '2001-06', '2001-07']...

Dataset: ideam/temperatura_minima.parquet (columna: fechaobservacion)
  Rango: 2001-02-17 17:39:03 a 2019-08-27 14:00:00
  Periodos esperados: 222 | Observados: 178
  ¡GAPS DETECTADOS! 45 periodos ausentes: ['2001-03', '2001-04', '2001-05', '2001-06', '2001-07']...

Dataset: dane/sipsa_insumos_precios.parquet (columna: fecha)
  Rango: 2018-12-01 00:00:00 a 2026-08-01 00:00:00
  Periodos esperados: 93 | Observados: 93
  Continuidad temporal perfecta (sin lagunas).

Dataset: dane/sipsa_alimentos_balanceados.parquet (columna: fecha)
  Rango: 2021-01-01 00:00:00 a 2026-08-01 00:00:00
  Periodos esperados: 68 | Observados: 68
  Continuidad temporal perfecta (sin lagunas).

Dataset: upra/upra_exportaciones_bioinsumos.parquet (co

In [13]:
# Celda 6: Matriz de Cardinalidad de Categorías y Variedades
print("=== MATRIZ DE CARDINALIDAD CATEGÓRICA ===")
for name, df in loaded_dfs.items():
    text_cols = [c for c in df.columns if df[c].dtype == "object" or df[c].dtype.name == "category"]
    if text_cols:
        c_mat = cardinality_matrix(df, categorical_cols=text_cols[:5])
        print(f"\nDataset: {name}")
        print(c_mat.to_string(index=False))

=== MATRIZ DE CARDINALIDAD CATEGÓRICA ===


In [14]:
# Celda 7: Resumen Ejecutivo de Calidad para la Capa Silver
print("=== CONCLUSIÓN DEL PERFILADO EDA ===")
print("1. Diagnóstico de missingness completado para fuentes crudas.")
print("2. Rangos de percentiles 1% y 99% disponibles para validación en contratos Pydantic V2.")
print("3. Insumos catalogados para normalización y sanitización en la Capa Silver (Sprint 3 y 4).")

=== CONCLUSIÓN DEL PERFILADO EDA ===
1. Diagnóstico de missingness completado para fuentes crudas.
2. Rangos de percentiles 1% y 99% disponibles para validación en contratos Pydantic V2.
3. Insumos catalogados para normalización y sanitización en la Capa Silver (Sprint 3 y 4).
